In [12]:
!nvidia-smi

from numba import cuda
import os
device = cuda.get_current_device()
print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y},{device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Thu Oct  8 08:22:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             34W /   70W |     111MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
import numpy as np
import time
from numba import cuda

def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
for n in sizes:
    cpu_t, gpu_tot, gpu_comp = benchmark_run(n)
    print(f"N={n:<10} | CPU: {cpu_t!s:<10} | GPU Total: {gpu_tot:.3f} ms | GPU Kernel: {gpu_comp:.3f} ms")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


N=100        | CPU: 0.06290599992553325 | GPU Total: 0.701 ms | GPU Kernel: 0.061 ms
N=1000       | CPU: 0.236590000099568 | GPU Total: 0.494 ms | GPU Kernel: 0.059 ms
N=10000      | CPU: 2.4676330000374946 | GPU Total: 0.518 ms | GPU Kernel: 0.058 ms
N=100000     | CPU: 22.764648000020316 | GPU Total: 0.704 ms | GPU Kernel: 0.067 ms
N=1000000    | CPU: 262.59210199998506 | GPU Total: 5.817 ms | GPU Kernel: 0.147 ms
N=5000000    | CPU: nan        | GPU Total: 12.269 ms | GPU Kernel: 0.271 ms
N=10000000   | CPU: nan        | GPU Total: 24.111 ms | GPU Kernel: 0.453 ms


In [14]:
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0

N_val = 1000
h_arr = np.zeros(N_val, dtype=np.float32)
d_arr = cuda.to_device(h_arr)
defective_kernel[4, 256](d_arr)
cuda.synchronize()

@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = cuda.blockIdx.x
        log_thread[g_idx] = cuda.threadIdx.x
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N_37 = 37
tpn = 8
bpg = (N_37 + tpn - 1) // tpn
log_b = cuda.device_array(bpg * tpn, dtype=np.int32)
log_t = cuda.device_array(bpg * tpn, dtype=np.int32)
log_g = cuda.device_array(bpg * tpn, dtype=np.int32)
log_a = cuda.device_array(bpg * tpn, dtype=np.int32)

inspect_threads_kernel[bpg, tpn](log_b, log_t, log_g, log_a, N_37)
cuda.synchronize()

log_g_host, log_b_host, log_t_host, log_a_host = log_g.copy_to_host(), log_b.copy_to_host(), log_t.copy_to_host(), log_a.copy_to_host()
for i in range(32, 40):
    status = "ACTIVE" if log_a_host[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g_host[i]:<10} | {log_b_host[i]:<10} | {log_t_host[i]:<10} | {status:<10}")

32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [15]:
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N_sum = 50000
h_data = np.ones(N_sum, dtype=np.float32)
d_data = cuda.to_device(h_data)

tp = 256
bp = (N_sum + tp - 1) // tp

for trial in range(1, 6):
    d_tot = cuda.to_device(np.zeros(1, dtype=np.float32))
    naive_sum_kernel[bp, tp](d_data, d_tot)
    cuda.synchronize()
    print(f"Trial {trial} Naive Sum:", d_tot.copy_to_host()[0])

d_tot_atomic = cuda.to_device(np.zeros(1, dtype=np.float32))
atomic_sum_kernel[bp, tp](d_data, d_tot_atomic)
cuda.synchronize()
print("Atomic Sum Output:", d_tot_atomic.copy_to_host()[0])

Trial 1 Naive Sum: 2.0
Trial 2 Naive Sum: 2.0
Trial 3 Naive Sum: 2.0
Trial 4 Naive Sum: 2.0
Trial 5 Naive Sum: 2.0
Atomic Sum Output: 50000.0


In [9]:
import math

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        d_canvas[y, x] = max(0.0, intensity)

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = ((WIDTH + 15) // 16, (HEIGHT + 15) // 16)

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()
print(f"Shape: {result_canvas.shape}, Center: {result_canvas[512, 512]:.3f}, Corner: {result_canvas[0, 0]:.3f}")

Shape: (1024, 1024), Center: 1.000, Corner: 0.000


In [11]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    h.update(cuda.get_current_device().name.encode('utf-8'))
    h.update(d_canvas.copy_to_host()[:10, 10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

STUDENT_ID = "230103124"
generate_lab_checksum(STUDENT_ID, d_canvas, 100000)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 3F7FE15DD3B7D692


'3F7FE15DD3B7D692'